# Some Exploratory Data Analysis Practice

## sklearn.neighbors

docs: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html#sklearn.neighbors.KNeighborsClassifier

examples: https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html#gallery-examples

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from matplotlib.colors import ListedColormap
import seaborn as sns
from sklearn import datasets, neighbors
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, Normalizer

# Practice area

Now let's try and train a simple (or not so) kNN classifier on a more complicated dataset.

In [ ]:
# If you are using colab, uncomment this cell

# !mkdir -p /content/sample_data/datasets
# !curl https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data > /content/sample_data/datasets/wine.csv

In [ ]:
feature_names = ['target', 'Alcohol', 'Malicacid', 'Ash', 'Alcalinity_of_ash', 'Magnesium', 'Total_phenols', 'Flavanoids', 'Nonflavanoid_phenols', 'Proanthocyanins', 'Color_intensity','Hue','0D280_0D315_of_diluted_wines','Proline']
dataset = pd.read_csv("/content/sample_data/datasets/wine.csv", header=None)
dataset.columns = feature_names
dataset.head()

# Simple EDA

Let's do Exploratory Data Analysis First

Let's analyze the basic stuff: shape and data types

In [ ]:
dataset.shape

In [ ]:
dataset.info()

Some datasets may contain NaNs due to errors or missing values, let's check for it:

In [ ]:
dataset.isnull().sum()

And also check for duplicates:

In [ ]:
dataset.duplicated().sum()

Let's make sure all the columns are numerical (they are):

In [ ]:
numerical_cols = dataset.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Found {len(numerical_cols)} numerical columns: {numerical_cols}")

Let's check for unique values

In [ ]:
dataset.nunique()

Let's see descriptive statistics:

In [ ]:
dataset.describe()

It is much more convenient to visualize the whole distributions instead of reading through numbers:

In [ ]:
for col in numerical_cols:
    plt.figure(figsize=(8, 5)) # Set the size for each figure
    sns.histplot(data=dataset, x=col, kde=True)
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency / Density')
    plt.show()

Let's also see through box plots and visualize outliers

In [ ]:
fig = plt.figure(figsize=(15, 15))
for idx, feature_name in enumerate(numerical_cols):
    plt.subplot(5, 3, idx + 1)
    sns.boxplot(x=dataset[feature_name], color='red')
    plt.title(f'boxplot for {feature_name}')
    plt.xlabel(feature_name)
plt.tight_layout()
plt.show()

These histogram plots may provide us with additional hints on the feature engineering: if some of the plots contain heavy distribution "tails" or look like some weird functions, you may engineer a way to transform the distribution in such a way, that the result is unimodal and looks like Normal distribution.

We may also observe the feature range. In our case features definitely have varying ranges and to use linear models or distance-based models, we'd rather center and scale the feature distributions independently.

It is also a good strategy to analyze feature cross correlation and highlight those features, that have high correlation with the target variable.

In [ ]:
corr = dataset.corr()
corr.style.background_gradient(cmap='coolwarm').format(precision=3)

It seems that for this dataset there are a lot of good features, with correlations of up to 0.85 ("Flavanoids"), that sum up and make it possible to create a really well performing model. 

In [ ]:
import matplotlib.pyplot as plt

class_correlations = corr.abs()['target'].drop('target')
class_correlations = class_correlations.sort_values(ascending=False)

# Create the bar plot
plt.figure(figsize=(10, 6))
plt.bar(class_correlations.index, class_correlations.values)
plt.xticks(rotation=45, ha='right')
plt.xlabel('Features')
plt.ylabel('Absolute Correlation with Class')
plt.title('Correlation of Features with Class')
plt.tight_layout()
plt.show()

# Building a model

Let's move on to building a model after our brief EDA in this case of a simple dataset

## Naiive approach

In [ ]:

X = dataset.drop('target', axis=1).to_numpy()
y = dataset['target'].to_numpy()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape)
print(X_test.shape)

In [ ]:
%time
log_model = LogisticRegression()

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

log_model.fit(X_train, y_train)
predictions = log_model.predict(X_test)
accuracy_score(y_test, predictions)

In [ ]:
%time
X_train_poly = PolynomialFeatures(degree=2).fit_transform(X_train)
X_test_poly = PolynomialFeatures(degree=2).fit_transform(X_test)

log_model.fit(X_train_poly, y_train)
predictions = log_model.predict(X_test_poly)
accuracy_score(y_test, predictions)

Maybe it would be possible to create a model with accuracy of 0.8 with just this one feature?

In [ ]:
X = dataset.iloc[:, 7].to_numpy().reshape(-1, 1)
y = dataset.iloc[:, 0].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

log_model.fit(X_train, y_train)
predictions = log_model.predict(X_test)
test_acc = accuracy_score(y_test, predictions)
print(f"test accuracy: {test_acc:.4f}")

Almost! But still it's very impressive that you could state the origin of the wine knowing only its Flavanoids number, with realistically good accuracy. Perhaps accuracy 0.8+ is possible with other models but it's out of scope of this task.

But can we *visually* determine the wine's origin, without doing any tests? Suppose we can determine Color intensity, Hue (which is color-related) and Alcohol which is usually stated on a bottle.

In [ ]:
X = dataset.iloc[:, [1,10,11]].to_numpy()
y = dataset.iloc[:, 0].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

log_model.fit(X_train, y_train)
predictions = log_model.predict(X_test)
test_acc = accuracy_score(y_test, predictions)
print(f"test accuracy: {test_acc:.4f}")

**Yes**, we can, and we can be *very* sure!

## A more serious approach
Let's implement feature pre-processing using `sklearn.pipeline` and run a proper cross-validation to determine the optimal set of parameters 

In [ ]:
from sklearn.pipeline import make_pipeline, Pipeline
base_model = Pipeline(steps=[
        ('poly', PolynomialFeatures(degree=2)),
        ('scaler', StandardScaler()),
        ('model', neighbors.KNeighborsClassifier())
    ]
)
base_model

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cross_vall_accuracies = cross_val_score(base_model, X_train, y_train, cv=skf, scoring='accuracy')
acc_mean = cross_vall_accuracies.mean()
acc_std = cross_vall_accuracies.std()
print(f"cross validated accuracy: {acc_mean:.3f} ± {acc_std:.3f}", )


In [ ]:
# naive way
best_score = 0.0
best_params = {}
for n_neighbors in range(1, 10):
    for weights in ['uniform', 'distance']:
        base_model = Pipeline(steps=[
                ('poly', PolynomialFeatures(degree=2)),
                ('scaler', StandardScaler()),
                ('model', neighbors.KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights))
            ]
        )
        cross_vall_accuracies = cross_val_score(base_model, X_train, y_train, cv=skf, scoring='accuracy')
        acc_mean = cross_vall_accuracies.mean()
        if acc_mean > best_score:
            best_score = acc_mean
            best_params = dict(n_neighbors=n_neighbors, weights=weights)
print("best params: ", best_params)
print(f"best score: {best_score:.3f}")

In [ ]:
final_model = Pipeline(steps=[
        ('poly', PolynomialFeatures(degree=2)),
        ('scaler', StandardScaler()),
        ('model', neighbors.KNeighborsClassifier(**best_params))
    ]
)
final_model

In [ ]:
final_model.fit(X_train, y_train)

In [ ]:
final_accuracy = final_model.score(X_test, y_test)
print(f"final TEST accuracy: {final_accuracy:.3f}")